# Translation quality scoring of the 500 Pass 1 items (run order items 3 and 4)

Author: Hrishin Debnath

**Why:** Steps 1 to 6 score every item, then Step 7 draws the 200-item hand sample. The CPU checks run anywhere; the three learned scorers (CometKiwi-22 primary, MetricX-24 secondary, LaBSE plus the option_swap check) need a GPU, so this notebook runs `run_all.py score --gpu` on Colab.

**What it needs:** a T4 runtime (Runtime > Change runtime type), a HuggingFace token, and the CometKiwi licence accepted once at huggingface.co/Unbabel/wmt22-cometkiwi-da (gated, no payment). Files to upload in cell 3: `run_all.py`, `bangla_qe_pipeline.py`, `step7_stats.py`, `500_translated_gptoss.json`, `backtranslations_gptoss120b_pass1_500.csv`.

**What it writes:** `results500/item_scores.csv`, `long_scores.csv`, `summary_by_translator.csv`, `throughput.json`, the rating sheets and keys. Scores are checkpointed in `results500/ckpt/`, so a disconnect costs one batch: re-run cell 5.

**Not verified yet:** the three learned scorers have not run on this data before. Cell 4 scores three test pairs first so a wrong package version fails in seconds, not after an hour.

In [ ]:
# Cell 1. GPU check and install (3 to 5 minutes). transformers is pinned last: MetricX-24 ran on 4.46.3 on 3 Sep 2026.
!nvidia-smi -L
!pip -q install sacrebleu pandas openpyxl scipy scikit-learn sentencepiece
!pip -q install "unbabel-comet>=2.2.0" sentence-transformers
!pip -q install "transformers==4.46.3"
!git clone -q https://github.com/google-research/metricx.git /content/metricx 2>/dev/null || echo "metricx already cloned"
import transformers, torch
print("transformers", transformers.__version__, "| torch", torch.__version__, "| cuda", torch.cuda.is_available())
print("If pip printed a dependency conflict about transformers, copy that message before going on.")

In [ ]:
# Cell 2. HuggingFace login. CometKiwi-22 is gated: accept the terms on its model page first, then paste a read token here.
from huggingface_hub import login
login()

In [ ]:
# Cell 3. Output folder and uploads.
import os
USE_DRIVE = True          # True: results live in Google Drive and survive a disconnect
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/bangla_gap/results500"
else:
    OUT = "/content/results500"
os.makedirs(OUT, exist_ok=True)

from google.colab import files
up = files.upload()       # run_all.py, bangla_qe_pipeline.py, step7_stats.py, 500_translated_gptoss.json, backtranslations_gptoss120b_pass1_500.csv
JSON = "500_translated_gptoss.json"
BT = "backtranslations_gptoss120b_pass1_500.csv"
for f in ["run_all.py", "bangla_qe_pipeline.py", "step7_stats.py", JSON, BT]:
    print(f, "OK" if os.path.exists(f) else "MISSING")
print("results go to", OUT)

In [ ]:
# Cell 4. Smoke test on three pairs (the third is a wrong option), then the 512-token cap check.
import time, json
import bangla_qe_pipeline as p

pairs = [("Drug reaction", "ওষুধের প্রতিক্রিয়া"),
         ("A 45-year-old man presents with fever of 38.5 C for 3 days.", "৪৫ বছর বয়সী একজন পুরুষ ৩ দিন ধরে ৩৮.৫ ডিগ্রি জ্বর নিয়ে এসেছেন।"),
         ("Drug reaction", "ফিজিক্যাল থেরাপি")]
src = [a for a, _ in pairs]; hyp = [b for _, b in pairs]
for name, fn in [("cometkiwi (0..1, higher better)", p.cometkiwi),
                 ("metricx error (0..25, lower better)", p.metricx_qe),
                 ("labse cosine", p.labse_cosine)]:
    t = time.time()
    print(f"{name}: {[round(x, 3) for x in fn(src, hyp)]}  ({time.time() - t:.0f} s including the model download)")

# How many segments exceed the CometKiwi encoder cap of 512 tokens per side. Those get cut silently.
import transformers
tok = transformers.AutoTokenizer.from_pretrained("xlm-roberta-large")      # same sentencepiece family as CometKiwi's encoder
recs = json.load(open(JSON, encoding="utf-8"))
n = over = mx = 0
for r in recs:
    hyp_c = p.clean_bangla(p.nfc(r["bangla"])[0])
    for segs in (p.segment(p.normalise_symbols(r["english"])), p.segment(hyp_c)):
        for s in segs:
            L = len(tok(s)["input_ids"]); n += 1; mx = max(mx, L); over += L > 512
print(f"{n} segments (English and Bangla). Longest {mx} tokens. {over} over the 512-token cap.")

In [ ]:
# Cell 5. Steps 1 to 6 on all 500 with the learned scorers (expect 20 to 40 minutes on a T4; re-run to resume).
!python run_all.py score --json "$JSON" --bt "$BT" --out "$OUT" --gpu --metricx-dir /content/metricx
print(open(f"{OUT}/throughput.json").read())

In [ ]:
# Cell 6. Step 8 with the hard flags only (no thresholds yet), then the Panel 1 style table.
!python run_all.py decide --out "$OUT"
!python run_all.py summary --out "$OUT" 

In [ ]:
# Cell 7. Step 7, the draw: 150 random + 25 worst ten percent by CometKiwi minimum + 25 largest CometKiwi vs MetricX disagreement.
!python run_all.py sample --out "$OUT" --n-random 150 --n-targeted 50

In [ ]:
# Cell 8. Download everything as one zip (the ckpt folder is left out).
parent, base = os.path.split(OUT.rstrip("/"))
!cd "$parent" && zip -qr results500.zip "$base" -x "*/ckpt/*"
files.download(os.path.join(parent, "results500.zip"))